# System Prompts

A **system prompt** customizes how Claude responds — its tone, style, and approach — before the conversation even starts. Instead of generic answers, you get behavior shaped to your use case.

**Example use case: a math tutor chatbot.** When a student asks *"How do I solve 5x + 2 = 3 for x?"*, a good tutor should:
- Give **hints** first, not the full answer
- Walk through the problem step by step
- Show similar worked examples

…and should **not** just hand over the answer or say "use a calculator."

How it works: you define the system prompt as a plain string and pass it as the `system=` argument to `create()`. It guides Claude, makes it respond like someone in that role would, and helps keep it on task.

> Kernel: **Python (anthropic-cert)**. Setup is repeated so this notebook runs standalone.

## Setup + a flexible `chat()`

The key upgrade this lesson makes: `chat()` now accepts an **optional** `system` prompt. A subtle detail — **the API does not accept `system=None`** — so we build a `params` dict and only add `system` when it's actually provided.

(As before, the one deviation from the course is `get_text(message)` instead of `message.content[0].text`, so a thinking block can't break it.)

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return get_text(message)


print("Ready. Flexible chat() loaded.")

Ready. Flexible chat() loaded.


## Seeing the difference

Same question, two behaviors. First **without** a system prompt — Claude just solves it.

In [2]:
question = "How do I solve 5x + 2 = 3 for x?"

messages = []
add_user_message(messages, question)

print(chat(messages))   # no system prompt -> full solution

# Solving 5x + 2 = 3

**Step 1: Isolate the term with x**

Subtract 2 from both sides:
$$5x + 2 - 2 = 3 - 2$$
$$5x = 1$$

**Step 2: Solve for x**

Divide both sides by 5:
$$x = \frac{1}{5}$$

**Check your answer** by plugging it back into the original equation:
$$5\left(\frac{1}{5}\right) + 2 = 1 + 2 = 3 ✓$$

So **x = 1/5** (or 0.2)


Now the **same question with the math-tutor system prompt** — Claude should give hints and guiding questions instead of the answer.

In [3]:
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

messages = []
add_user_message(messages, question)

print(chat(messages, system=system))   # tutor persona -> hints, not answers

I'd be happy to guide you through this! Let's think about it step by step.

Right now you have:
$$5x + 2 = 3$$

**First question for you:** What operation is being done to the term with x (that is, to 5x)? In other words, what's happening to 5x in this equation?

Once you identify that, think about what step you could take to start isolating the x term.


## Why this matters

The `system` prompt is separate from the `messages` list — it's not a user or assistant turn, it's standing guidance that applies to the whole conversation. Passing it as an optional parameter (rather than hard-coding) makes `chat()` reusable: the same function powers a plain assistant, a math tutor, a code reviewer, etc., just by swapping the string.

You'll reuse this `chat(messages, system=None)` signature throughout the rest of the course.